# MIF 2026 — auditoria dos cruzamentos estratégicos

## tl;dr

O encerramento muda para 5K (43,79% da fase); SC e RS concentram mais de 60% das próprias vendas no fechamento; Top 10 canais respondem por 85,03% do valor; e os 115 canais em testar/renegociar somam apenas 5,76% do bruto.

## Context & Methods

Companheiro reproduzível do relatório editorial. Usa somente agregados anônimos do snapshot final fechado e valida cruzamentos de fase, distância, lote, estado, canal e produto.

### Key Assumptions

Fases e lotes descrevem o mesmo ciclo e são colineares; produtos podem se sobrepor; especialistas exigem ao menos 30 inscrições no canal e 10 na célula.

## Data

Fontes locais: `general.json`, `strategy.json`, `channels/index.json` e `manifest.json` em `modular_dist/`.

In [ ]:
import json
from pathlib import Path
import pandas as pd

root = Path.cwd()
data_dir = root / '_codex/analyses/mif_2026_channels/modular_dist'
general = json.loads((data_dir / 'general.json').read_text(encoding='utf-8'))
strategy = json.loads((data_dir / 'strategy.json').read_text(encoding='utf-8'))
channels = json.loads((data_dir / 'channels/index.json').read_text(encoding='utf-8'))
manifest = json.loads((data_dir / 'manifest.json').read_text(encoding='utf-8'))
print(strategy['meta'])

In [ ]:
paid = general['overview']['paid_registrations']
phase_modality = pd.DataFrame(strategy['datasets']['phase_modality'])
assert int(phase_modality['paid_registrations'].sum()) == paid == 15713
assert len(channels['channels']) == 156
assert 'strategy.json' in manifest['artifacts']
assert all(key not in strategy for key in ('observations', 'registration_cube', 'product_cube'))
print(f'Reconciliação OK: {paid:,} inscrições; {len(channels["channels"])} canais; estratégia compacta sem cubos detalhados.')

## Results

### Calendário de ativação

In [ ]:
phase_playbook = pd.DataFrame(strategy['insights']['phase_playbook'])
phase_playbook[['phase', 'paid_registrations', 'event_share_pct', 'lead_modality', 'lead_modality_share_pct', 'lead_lot', 'lead_state', 'lead_channel']]

### Playbook por distância

In [ ]:
distance_playbook = pd.DataFrame(strategy['insights']['distance_playbook'])
distance_playbook[['modality', 'paid_registrations', 'event_share_pct', 'peak_phase', 'lead_lot', 'lead_state', 'volume_channel', 'specialist_channel', 'specialist_index', 'specialist_cell']]

### Timing territorial e portfólio

In [ ]:
state_timing = pd.DataFrame(strategy['insights']['state_timing'])
print(state_timing.to_string(index=False))
portfolio = strategy['insights']['channel_portfolio']
print(pd.DataFrame(portfolio['recommendation_groups']).to_string(index=False))
print({key: value for key, value in portfolio.items() if key != 'recommendation_groups'})

### Produtos adicionais por distância

In [ ]:
products = pd.DataFrame(strategy['insights']['product_opportunities'])
products[['modality', 'paid_registrations', 'leading_product', 'leading_product_registrations', 'leading_product_take_rate_pct']]

## Takeaways

Síntese editorial reproduzida abaixo diretamente do artefato validado.

In [ ]:
for item in strategy['insights']['executive_takeaways']:
    print(f"{item['title']}\n  Evidência: {item['evidence']}\n  Implicação: {item['implication']}\n")
print('Limitações:')
for caveat in strategy['caveats']:
    print('-', caveat)